# Chapter 2 - TF-IDF

tf - how often a word appear in a doc.

df - how many documents contain it.

### Step 1 - Set up

In [1]:
import numpy as np
import pandas as pd

In [2]:
docs = [
    "the discount was reduced",
    "we lowered our pricing",
    "the customer filed a complaint",
    "the discount policy was the best discount",
]

vocab = sorted({w for d in docs for w in d.split()})
word_to_idx = {w: i for i, w in enumerate(vocab)}

def bag_of_words(doc, vocab, word_to_idx):
    v = np.zeros(len(vocab))
    for w in doc.split():
        if w in word_to_idx:
            v[word_to_idx[w]] += 1
    return v

X = np.array([bag_of_words(d, vocab, word_to_idx) for d in docs])
pd.DataFrame(X.astype(int), columns=vocab, index=[f"doc{i+1}" for i in range(4)])

,a,best,complaint,customer,discount,filed,lowered,our,policy,pricing,reduced,the,was,we
doc1,0,0,0,0,1,0,0,0,0,0,1,1,1,0
doc2,0,0,0,0,0,0,1,1,0,1,0,0,0,1
doc3,1,0,1,1,0,1,0,0,0,0,0,1,0,0
doc4,0,1,0,0,2,0,0,0,1,0,0,2,1,0


### Step 2- Compute df

In [3]:
N = len(docs)
df = (X > 0).sum(axis=0)
df

array([1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 3, 2, 1])

In [4]:
pd.DataFrame({"word": vocab, "df":df}).sort_values("df", ascending=False)

,word,df
11,the,3
4,discount,2
12,was,2
0,a,1
1,best,1
2,complaint,1
3,customer,1
5,filed,1
6,lowered,1
7,our,1


### Step 3 - Your first instinct, tested

In [5]:
idf_naive = N/df
pd.DataFrame({"word": vocab, "idf_naive": idf_naive}).sort_values("idf_naive", ascending=False)

,word,idf_naive
0,a,4.000000
1,best,4.000000
2,complaint,4.000000
3,customer,4.000000
5,filed,4.000000
6,lowered,4.000000
7,our,4.000000
8,policy,4.000000
9,pricing,4.000000
10,reduced,4.000000


### Step 4 - Scale it up

In [6]:
N_big = 10000
for df_val in [1, 10, 100, 1000, 5000, 9000]:
    print(f"df={df_val:5d}  N/df={N_big/df_val:10.2f}")

df=    1  N/df=  10000.00
df=   10  N/df=   1000.00
df=  100  N/df=    100.00
df= 1000  N/df=     10.00
df= 5000  N/df=      2.00
df= 9000  N/df=      1.11


A word in 1 doc is having score 10000 and 10 have score 1000. Where as 5000 and 9000 frequent useless words got different scores.

### Step 5 - The fix

In [7]:
for df_val in [1, 10, 100, 1000, 5000, 9000]:
    raw = N_big / df_val
    print(f"df={df_val:5d}   N/df = {raw:10.2f}   log(N/df) = {np.log(raw):6.2f}")

df=    1   N/df =   10000.00   log(N/df) =   9.21
df=   10   N/df =    1000.00   log(N/df) =   6.91
df=  100   N/df =     100.00   log(N/df) =   4.61
df= 1000   N/df =      10.00   log(N/df) =   2.30
df= 5000   N/df =       2.00   log(N/df) =   0.69
df= 9000   N/df =       1.11   log(N/df) =   0.11


In [9]:
np.log(1)

np.float64(0.0)

### Step 6 - The +1

When df = 0 > N/df crashes. So we add 1 to denominator.

idf = log (N/1+df)

### Step 7 - Build it

In [10]:
idf = np.log(N/(1+df))
pd.DataFrame({"word": vocab, "idf": idf}).sort_values("idf", ascending=False)

,word,idf
0,a,0.693147
1,best,0.693147
2,complaint,0.693147
3,customer,0.693147
5,filed,0.693147
6,lowered,0.693147
7,our,0.693147
8,policy,0.693147
9,pricing,0.693147
10,reduced,0.693147


In [12]:
X

array([[0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 1., 0.],
       [0., 0., 0., 0., 0., 0., 1., 1., 0., 1., 0., 0., 0., 1.],
       [1., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 2., 0., 0., 0., 1., 0., 0., 2., 1., 0.]])

In [13]:
tfidf = X * idf
pd.DataFrame(tfidf, columns=vocab, index=[f"doc{i+1}" for i in range(4)]).round(2)

,a,best,complaint,customer,discount,filed,lowered,our,policy,pricing,reduced,the,was,we
doc1,0.00,0.00,0.00,0.00,0.29,0.00,0.00,0.00,0.00,0.00,0.69,0.0,0.29,0.00
doc2,0.00,0.00,0.00,0.00,0.00,0.00,0.69,0.69,0.00,0.69,0.00,0.0,0.00,0.69
doc3,0.69,0.00,0.69,0.69,0.00,0.69,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00
doc4,0.00,0.69,0.00,0.00,0.58,0.00,0.00,0.00,0.69,0.00,0.00,0.0,0.29,0.00


Lookat the scores now, the obvious words the and was tend to 0. This is what we want to do.

Now lets check the documents similarity. First using Euclidean distance.

### Step 8 - Euclidean distance

In [15]:
a = np.array([1, 1, 0, 0])          # short document
b = np.array([10, 10, 0, 0])        # same words, said 10x more
c = np.array([0, 0, 1, 1])          # completely different words

print("euclidean a-b:", np.linalg.norm(a - b))
print("euclidean a-c:", np.linalg.norm(a - c))

euclidean a-b: 12.727922061357855
euclidean a-c: 2.0


By looking at the a and b docs we can see it is more similar but the distance is saying it is further apart.

But do vectors a and b point in the same direction, Yes! Inorder to calculate direction we use cosine similarity 


cos theta = a.b / ||a|| ||b||,

a.b is the dot product and || || is norm

### Step 9 - Cosine similarity

In [16]:
def cosine(a, b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))

print("cosine a-b:", cosine(a, b))
print("cosine a-c:", cosine(a, c))

cosine a-b: 0.9999999999999998
cosine a-c: 0.0


### Step 10 - Cos similarity on our docs

In [20]:
n = len(docs)
sim = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim[i,j] = cosine(tfidf[i], tfidf[j])

labels = [f"docs{i+1}" for i in range(n)]
pd.DataFrame(sim, columns=labels, index=labels).round(3)

,docs1,docs2,docs3,docs4
docs1,1.000,0.0,0.0,0.263
docs2,0.000,1.0,0.0,0.000
docs3,0.000,0.0,1.0,0.000
docs4,0.263,0.0,0.0,1.000


### Step 11: The flaw

In [ ]:
i = 0   # "the discount was reduced"
j = 1   # "we lowered our pricing"

print(f"cosine similarity between doc{i+1} and doc{j+1}: {sim[i,j]:.3f}")

cosine similarity between doc1 and doc2: 0.000


We know both docs means same thing but it scores 0. TF-IDF dont know discount and pricing are identical.